In [ ]:
import numpy as np
import random

In [ ]:
class Problem:

    def __init__(self, states, initial, goal, actions, transition_model, cost):
        self.states = states                #estados possíveis
        if initial not in states:           #verifica se o estado inicial é um estado possível
            self.states.append(initial)     #caso não seja, adiciona o estado inicial aos estados possíveis
        self.initial = initial              #estado inicial do problema
        if goal not in states:              #verifica se o estado objetivo é um estado possível
            self.states.append(goal)        #caso não seja, adiciona o estado objetivo aos estados possíveis
        self.goal = goal                    #estado(s) objetivo do problema
        self.actions = actions              #ações possíveis
        self.transition_model = transition_model
        self.cost = cost
    def get_actions(self, state):
        return self.actions[state]
    def result(self, state, action):
        return self.transition_model[state][action]
    def goal_test(self, state):
        return state == self.goal
    def action_cost(self, state1, action, state2):
        if action in self.actions[state1] and state2 == self.result(state1, action):
            return self.cost[state1][state2]
        else:
            return -1

In [ ]:
class Node:
    
    def __init__(self, state, parent = None, action = None, path_cost = 0):
        self.state = state          # o estado ao qual o nó corresponde; (str)
        self.parent = parent        # o nó da árvore que gerou este nó; (Node)
        self.action = action        # a ação executada para gerar este nó; (str)
        self.path_cost = path_cost  # o custo do caminho do nó inicial até este nó. (int)

class Switchs(Node):
    def __init__(self, val, parent = None, action = None, path_cost = 1, n = 16):
        self.n = n
        state = {}
        state['dec'] = val
        state['bin'] = Switchs.int_to_bin_list(val, self.n)
        super().__init__(state, parent, action, path_cost)
        self.xobj = np.array(range(1, 17), dtype=np.float64) # 16 pontos
        # definindo o vetor de pesos
        w = np.zeros(16, dtype=np.uint8)
        # definindo os pesos para os 8 primeiros pontos
        w[:8] = np.array([1, 0, 1, 0, 1, 0, 0, 1], dtype=np.uint8)
        self.yobj = np.random.uniform(-50, 50, 16)          # 16 valores aleatórios
        # definindo os valores de y para os 8 primeiros pontos
        self.yobj[:8] = np.array([2.0, -10.0, 3.0, -20.0, 1.0, -30.0, -40.0, 4.0], dtype=np.float64)
        self.yobj = np.array([Switchs.pLagrangeW(xi, self.xobj, self.yobj, w) for xi in self.xobj])
    
    def int_to_bin_list(val, n):
        binList = [int(digit) for digit in bin(val)[2:]]
        binList = [0]*(n-len(binList)) + binList
        return binList
    
    def bin_list_to_int(binList):
        return int(''.join(map(str, binList)), 2)
    
    def distQuad(y_inter, y):
        n = len(y)
        return np.sqrt((np.sum((y_inter - y)**2))/n)
    
    def pLagrange(x, x_i, y_i):
        n = len(x_i)
        L = np.zeros(n)
        for i in range(n):
            L[i] = np.prod([(x - x_i[j])/(x_i[i] - x_i[j]) for j in range(n) if i != j])
        #print(L.shape)
        #print(y_i.shape)
        return np.dot(y_i, L)
    
    def pLagrangeW(x, x_i, y_i, w):
        # Vamos gerar o polinômio de Lagrange apenas com os pontos que possuem peso 1
        x_ = x_i[w == 1]
        y_ = y_i[w == 1]
        return Switchs.pLagrange(x, x_, y_)
    
    def custo(self):
        w = np.array(self.state['bin'])
        y_inter = y_inter = np.array([Switchs.pLagrangeW(xi, self.xobj, self.yobj, w) for xi in self.xobj])
        return Switchs.distQuad(y_inter, self.yobj)

    def turnOnOff(ind, swc):
        if swc < 0 or swc >= ind.n:
            return False
        ind.state['bin'][swc] = 1 - ind.state['bin'][swc]
        ind.state['dec'] = Switchs.bin_list_to_int(ind.state['bin'])
        return True

states = []
#for i in range(1,2**16):
#    states.append(Switchs(i))

initial = random.randint(1,2**16)
initial = Switchs(initial)
goal = None
actions = list(range(16))
switchs16 = Problem(states, initial, goal, actions, Switchs.turnOnOff, cost = 1)

In [ ]:
'''
function HILL-CLIMBING(problem) returns a state that is a local maximum 
    current←problem.INITIAL
    while true do
        neighbor ← a highest-valued successor state of current
        if VALUE(neighbor) ≤ VALUE(current) then return current 
        current ← neighbor

'''

def highest_value_successor(current, problem):
    neighbor = Switchs(current.state['dec'])
    problem.transition_model(neighbor, 0)
    value = neighbor.custo()
    for action in range(1,16):
        otherNeighbor = Switchs(current.state['dec'])
        problem.transition_model(otherNeighbor, action)
        otherValue = otherNeighbor.custo()
        if otherValue > value:
            neighbor = otherNeighbor
            value = otherValue
    return neighbor

def hill_climbing(problem):
    current = problem.initial
    print(current.state['dec'], current.custo())
    while True:
        neighbor= highest_value_successor(current, problem)
        if neighbor.custo() <= current.custo():
            return current
        current = neighbor
        print(current.state['dec'], current.custo())

In [24]:
max = hill_climbing(switchs16)
print(max.state['dec'], max.state['bin'], max.custo())

35349 7.862278160115583e-15
35351 1.6261251540294854e-13
2583 1.7206867944883556e-12
2591 2.077539642798189e-11
2623 1.2064921044030978e-10
2751 3.0405238551379184e-10
2815 7.087642680130499e-10
767 1.265904525495192e-09
1023 1.914639367246231e-09
511 1.9208208648120944e-09
1535 1.9867359736520174e-09
1535 [0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1] 1.9867359736520174e-09


37365 3.1131365927807205e-13
4597 7.59393093506005e-12
4605 7.178203607481793e-11
4607 5.726707595449234e-10
511 1.9208208648120944e-09
1535 1.9867359736520174e-09
